# Bằng chứng KLTN: báo cáo chẩn đoán RQ1–RQ2

Snapshot08/11/2026. **Đây là pilot/chẩn đoán đã lưu, không kết quả cuối của ba RQ.**
Notebook chỉ đọc các artifact đóng gói, kiểm hash và dựng bảng/sơ đồ. Không train,
chạy simulator, truy vấn Junyi/Holdout hoặc kết nối cloud.

Outputs được tính thật bằng `.venv` qua bộ dựng stdlib; chưa kiểm bằng Jupyter
kernel. Không cần cài thư viện để mở đọc outputs. Khi chạy lại trong notebook,
kernel cần IPython; không cài thư viện global. Số events/tests không là số mẫu độc lập.


In [1]:
from pathlib import Path
import hashlib
import json

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / 'research_tracking/reports/THESIS_RQ12_DIAGNOSTIC_RESULTS_DRAFT_20261108.sources.json').is_file()), None)
if ROOT is None:
    raise RuntimeError("Mở notebook bên trong repository KLTN")
builder = ROOT / 'scripts/research/build_thesis_evidence_notebook.py'
builder_payload = builder.read_bytes()
if hashlib.sha256(builder_payload).hexdigest() != '5f49bb44138087ebd2d783fa6d8bca7d1ec028bcc970ebde2cfd3e0500b94952':
    raise RuntimeError("Builder đã thay đổi; cần regenerate notebook theo phiên bản mới")
helpers = {'__name__': 'kltn_verified_report_builder', '__file__': str(builder)}
exec(compile(builder_payload, str(builder), 'exec'), helpers)
verified = helpers['load_verified_evidence'](ROOT)
source_map, cache = verified['source_map'], verified['cache']
table = helpers['table']
def show_html(text):
    capture = globals().get('_REPORT_HTML_CAPTURE')
    if capture is not None:
        capture(text)
    else:
        from IPython.display import HTML, display
        display(HTML(text))
print(json.dumps({k:v for k,v in verified.items() if k not in ('source_map','cache')}, ensure_ascii=False))


{"verified_inputs": 28, "verified_pointers": 104, "verified_claims": 14}


## 1. Sơ đồ đọc kết quả, không phải một funnel nhân quả

Sơ đồ dưới mô tả thứ tự native Paper2 được chẩn đoán. P của bài đã chọn được
tính sau selection; không đặt Pserved ở đầu một chuỗi nhân quả dẫn đến selection.
P của current query trong fixed-stream cũng không phải toàn vector trạng thái.


In [2]:
stages = ["Trạng thái/score concept S", "Mastery M + ready R",
          "Candidates C / balanced B / keys", "Chọn concept và item A",
          "Dự đoán Pserved", "Interact rồi update"]
show_html(table(["Bước", "Thứ tự native Paper2"], [(i+1, stage) for i,stage in enumerate(stages)]))
print("S → M/R → C/B/keys → A → Pserved → interact/update")
print("Đây là sơ đồ nguồn; chưa chứng minh causal mediation hoặc graph edge-only effect.")


Bước,Thứ tự native Paper2
1,Trạng thái/score concept S
2,Mastery M + ready R
3,Candidates C / balanced B / keys
4,Chọn concept và item A
5,Dự đoán Pserved
6,Interact rồi update


S → M/R → C/B/keys → A → Pserved → interact/update
Đây là sơ đồ nguồn; chưa chứng minh causal mediation hoặc graph edge-only effect.


## 2. RQ1: thay đổi số có truyền thành thay đổi quyết định?

C01–C03. Mỗi comparison có1.080 paired pre-update snapshots, từ3 seed blocks ×
3 profiles ×120steps. Events và profiles cùng seed có thể phụ thuộc nhau. M là
event có ít nhất một concept đổi classification; A là **concept preview**, không
served action. Các cột là số đếm song song, không funnel với mẫu số giảm dần.


In [3]:
transitions = cache['research_tracking/runs/cpu-rq1-transitions-20261108-01/analysis.json']
pooled = [g for g in transitions['fixed_stream']['groups'] if g['scope'] == 'pooled']
fields = ['event_pairs','changed_probability_events','changed_mastery_events_any_concept',
          'changed_ready_events','changed_action_preview_events']
show_html(table(['Condition','Paired events','P đổi','M đổi','R đổi','A preview đổi'],
                [[g['candidate']] + [g[k] for k in fields] for g in pooled]))
full = next(g for g in pooled if g['candidate'] == 'graph_package')
selected_conditions = [r for r in full['conditional_tables']
                       if r['first'] == 'p_changed' and r['first_changed']
                       and r['second'] == 'a_changed' and not r['second_changed']]
show_html(table(['Điều kiện','Số đếm','Mẫu số','Tỷ lệ mô tả'],
                [('P đổi nhưng concept preview giữ', r['count'], r['conditional_denominator'],
                  r['conditional_fraction']) for r in selected_conditions]))
print("Package graph có thay đổi ngoài cạnh; chưa cô lập causal contribution của prerequisite edges.")


Condition,Paired events,P đổi,M đổi,R đổi,A preview đổi
graph_package,1080,964,90,60,22
empty_graph_package,1080,37,0,0,0


Điều kiện,Số đếm,Mẫu số,Tỷ lệ mô tả
P đổi nhưng concept preview giữ,944,964,0.97925311


Package graph có thay đổi ngoài cạnh; chưa cô lập causal contribution của prerequisite edges.


## 3. RQ1: giải thích first-break trong closed loop

C14 là reconstruction từ source hiện có, không phục hồi nguyên runner lịch sử.
Chỉ phân tích tại common observed prefix trước phân kỳ; sau đó không paired ΔP/loss
giữa hai histories khác nhau. Các nhóm dưới là giải thích key chọn **CID**, không
toàn bộ lựa chọn item hoặc chứng minh learning benefit. Không thêm independent N.


In [4]:
ranking = [row for row in source_map['derived_arithmetic'] if row['claim'] == 'C14']
show_html(table(['Condition','Cùng M/R/C/B, khác score chọn CID','Membership khác',
                'Không tuple first-break','Comparison count'],
               [(r['candidate'], r['counts']['same_membership_score_key'],
                 r['counts']['membership_diff'], r['counts']['no_tuple_break'], r['denominator']) for r in ranking]))


Condition,"Cùng M/R/C/B, khác score chọn CID",Membership khác,Không tuple first-break,Comparison count
graph_package,8,1,0,9
empty_graph_package,2,0,7,9


## 4. RQ2 — Trap1: retention có cung cấp thông tin mới?

C08. Hệ số dưới chỉ đúng cho no-practice sau T, homogeneous decay và nonnegative
gain trong fixture đã kiểm. Final state vẫn phụ thuộc lịch luyện trước đó. Không
suy mọi retention model đều không phân biệt policy.


In [5]:
retention = cache['research_tracking/runs/cpu-rq12-20261107-03/retention.json']
show_html(table(['Thông số/kiểm tra','Giá trị'], [(k,retention[k]) for k in
               ['true_forget_rate','retention_days','scaling_factor','absolute_error']]))
print("Trong điều kiện đã nêu: retained_gain14 = scaling_factor × final_net_gain.")


Thông số/kiểm tra,Giá trị
true_forget_rate,0.06
retention_days,14
scaling_factor,0.43171052
absolute_error,1.7053026e-13


Trong điều kiện đã nêu: retained_gain14 = scaling_factor × final_net_gain.


## 5. RQ2 — Trap2 và Trap3: điều kiện và confounding

Factorial:10 seed blocks/profile; contrast là tương tác điều kiện simulator,
chưa là causal ability-only DiD. Junyi: phân tích order-only trên inner-train,
không dùng correctness hoặc Holdout. Predictive/logged-order association không
chứng minh lợi ích giảng dạy của graph.


In [6]:
conditions = cache['research_tracking/runs/rq2-conditions-evidence-20261108-01/conditions.json']
show_html(table(['Profile','Seed blocks','Exposure fraction','Mean interaction','Signs'],
               [(r['profile'],r['seed_blocks'],r['pooled_mechanism_exposure_fraction'],
                 r['mean_lambda_edge_interaction'],r['interaction_signs'])
                for r in conditions['factorial']['profiles']]))
junyi_claim = next(c for c in source_map['claims'] if c['id'] == 'C11')
show_html(table(['Nguồn order-only','Giá trị','JSON pointer'],
               [(Path(r['path']).name,r['value'],r['json_pointer']) for r in junyi_claim['evidence'][:5]]))
for limit in conditions['limits_vi']:
    print("Giới hạn:", limit)


Profile,Seed blocks,Exposure fraction,Mean interaction,Signs
PG,10,0,0,"{""positive"": 0, ""negative"": 0, ""arithmetic_zero"": 10}"
PY,10,0.79958333,0.034017258,"{""positive"": 7, ""negative"": 3, ""arithmetic_zero"": 0}"
PYH,10,0.49875,0.053528587,"{""positive"": 7, ""negative"": 3, ""arithmetic_zero"": 0}"


Nguồn order-only,Giá trị,JSON pointer
order_summary.json,256,/learners
order_summary.json,12575,/events
manifest.json,0,/outer_fold
manifest.json,inner-train learners only,/scope
manifest.json,300,/max_events_per_learner


Giới hạn: Không causal ability effect, final efficacy, equivalence, p-value hoặc formal CI.
Giới hạn: Không pooling đơn vị seed, event, edge hoặc learner giữa hai hệ.
Giới hạn: Không per-seed mechanism exposure hoặc per-learner contrasts trong selected inputs.
Giới hạn: Profile dynamics/initialization và graph correlation của nguồn được giữ; không giả ability-only intervention.


## 6. Phát biểu nào được phép đưa vào báo cáo?

- Có thể: chẩn đoán cơ chế, số đếm/đơn vị rõ ràng, conditional retention identity,
  nhiễu practice order và giới hạn đối chứng.
- Chưa thể: graph cải thiện học tập nhân quả, CG-AKT vượt baseline, hoặc RQ hoàn tất.
- Các PASS phần mềm đặt ở phụ lục tái lập; không dùng số tests làm hiệu quả mô hình.
- Chưa có original Paper1 revision; phân biệt current-source diagnosis và reproduction.

Mỗi C-ID bên dưới trỏ tới source map có đường dẫn, SHA và JSON Pointer/giá trị.


In [7]:
show_html(table(['Claim ID','Phát biểu/nhóm chứng cứ','Số pointers'],
               [(c['id'],c['claim_vi'],len(c['evidence'])) for c in source_map['claims']]))
print("Source map:", ROOT / helpers['MAP_PATH'])
print("Source map SHA:", helpers['MAP_SHA256'])
print("Bản thảo:", ROOT / source_map['document'])


Claim ID,Phát biểu/nhóm chứng cứ,Số pointers
C01,Fixed-stream units/semantics; M-event khác concept denominator,6
C02,P/S/M/R/A full và empty package totals; S không calibrated mastery,3
C03,Conditional event-level cooccurrence; không nested funnel,1
C04,Stored deterministic examples; P không fullstate,3
C05,Loss/Brier pooled và all seed-profile deltas; PG xấu hơn ở3seeds; seed20000score changes,12
C06,Closed-loop served action audit và full chain missing,1
C07,Paper1 source counterexample và chưa tìm exact revision trong scope audit,5
C08,Retention identity có điều kiện; không đồng nhất mọi metric,1
C09,Factorial conditional interaction/source treatment/exposure; PG zero không equivalence,8
C10,Profile difference LOO sensitivity không CI,1


Source map: D:\Dev\KLTN\research_tracking\reports\THESIS_RQ12_DIAGNOSTIC_RESULTS_DRAFT_20261108.sources.json
Source map SHA: fbfa712e1eaf04f05fee5a33c577393478ba7f5a766af2a836d4598a2ed73bbc
Bản thảo: D:\Dev\KLTN\research_tracking\reports\THESIS_RQ12_DIAGNOSTIC_RESULTS_DRAFT_20261108.md
